In [ ]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from torch.cuda.amp import autocast  # 对于支持 CUDA 的设备，使用 amp
# Memory optimization imports
import torch.nn.utils.prune as prune
from typing import Optional
import os
import gc

def clear_memory():
    """Clear GPU/MPS memory and run garbage collection"""
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    gc.collect()

def get_memory_usage():
    """Get current memory usage"""
    if torch.backends.mps.is_available():
        allocated = torch.mps.current_allocated_memory() / 1024**3  # GB
        reserved = torch.mps.driver_allocated_memory() / 1024**3   # GB
        print(f"MPS Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
        return allocated, reserved
    return 0, 0

def setup_memory_efficient_training():
    """Setup memory efficient training environment"""
    # Set environment variables for memory optimization
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'  # Disable memory caching
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # Enable fallback to CPU
    
    # Clear memory at start
    clear_memory()

def generate_responses(model, tokenizer, user_message, system_message=None, 
                       max_new_tokens=50):  # Reduced from 100 to save memory
    """Generate response with memory optimization"""
    # Format chat using tokenizer's chat template
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    
    messages.append({"role": "user", "content": user_message})
        
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    
    # Move to device only when needed
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        # Use smaller memory footprint generation
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=False,  # Disable KV cache to save memory
        )
    
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Clear intermediate tensors
    del inputs, outputs, generated_ids
    clear_memory()
    
    return response

def test_model_with_questions(model, tokenizer, questions, 
                              system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        try:
            response = generate_responses(model, tokenizer, question, 
                                          system_message)
            print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"Memory error for question {i}, skipping...")
                clear_memory()
            else:
                raise e



def load_model_and_tokenizer(model_name, use_gpu=False):
    """Load model with memory optimization options"""
    
    print(f"Loading model: {model_name}")
    
    # Load tokenizer first (smaller memory footprint)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Load model with optimizations
    model_kwargs = {
        "torch_dtype": torch.float32,  # Use half precision
        "low_cpu_mem_usage": True,     # Reduce CPU memory usage during loading
    }
    
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    
    # Move to appropriate device
    if use_gpu and torch.backends.mps.is_available():
        model.to("mps")
        print("Model moved to MPS")
    elif use_gpu and torch.cuda.is_available():
        model.to("cuda")
        print("Model moved to CUDA")
    else:
        print("Using CPU")
    
    # Setup chat template if not exists
    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""
    
    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
        
    return model, tokenizer

questions = [
    "Give me a 1-sentence introduction of LLM.",
    "Calculate 1+1-1",
    "What's the difference between thread and process?"
]


print("Testing trained model...")
setup_memory_efficient_training()
model, tokenizer = load_model_and_tokenizer("./model/Qwen/Qwen3-0.6B-Base-SFT", use_gpu=True)
try:
    model.to("mps")  # Move to CPU for inference to save MPS memory
    test_model_with_questions(model, tokenizer, questions, 
                              title="Model (After SFT) Output")
except Exception as e:
    print(f"Testing error: {e}")
    print("Model testing skipped due to memory constraints")


Testing trained model...


NameError: name 'gc' is not defined